In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import os 
import torch
from torch.autograd import Variable
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torchvision import transforms
from PIL import Image
from einops import rearrange, repeat, reduce
import numpy as np
from tqdm import tqdm
import sys

sys.path.append('..')
from utils import resize_to_max_size, preprocess_image, preprocess_greyscale_image, postp, LuminanceRemapper, MaskedMSELoss, resize_scalar_field, receptive_resize_scalar_field, minpool_resize_scalar_field

base_dir = os.path.join(os.getcwd(), "..")
content_dir = os.path.join(base_dir, 'images/content')
style_dir = os.path.join(base_dir, "images/styles")
model_dir = os.path.join(base_dir, "SANet/models")
scalar_field_dir = os.path.join(base_dir, "images/scalar_fields")

print("pytorch version: ", torch.__version__)
print("pytorch cuda version: ", torch.version.cuda)
print("cuda is available: ", torch.cuda.is_available())
print("number of available cpus: ", os.cpu_count())
print("current working directory: ", os.getcwd())

In [ ]:
#vgg definition that conveniently let's you grab the outputs from any layer
class VGG(nn.Module):
    def __init__(self, pool='max'):
        super(VGG, self).__init__()
        #vgg modules
        self.conv1_1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv1_2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv2_1 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.conv2_2 = nn.Conv2d(128, 128, kernel_size=3, padding=1)
        self.conv3_1 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.conv3_2 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_3 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv3_4 = nn.Conv2d(256, 256, kernel_size=3, padding=1)
        self.conv4_1 = nn.Conv2d(256, 512, kernel_size=3, padding=1)
        self.conv4_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv4_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_1 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_2 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_3 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        self.conv5_4 = nn.Conv2d(512, 512, kernel_size=3, padding=1)
        if pool == 'max':
            self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.MaxPool2d(kernel_size=2, stride=2)
        elif pool == 'avg':
            self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool3 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool4 = nn.AvgPool2d(kernel_size=2, stride=2)
            self.pool5 = nn.AvgPool2d(kernel_size=2, stride=2)
            
    def forward(self, x, out_keys):
        out = {}
        out['r11'] = F.relu(self.conv1_1(x))
        out['r12'] = F.relu(self.conv1_2(out['r11']))
        out['p1'] = self.pool1(out['r12'])
        out['r21'] = F.relu(self.conv2_1(out['p1']))
        out['r22'] = F.relu(self.conv2_2(out['r21']))
        out['p2'] = self.pool2(out['r22'])
        out['r31'] = F.relu(self.conv3_1(out['p2']))
        out['r32'] = F.relu(self.conv3_2(out['r31']))
        out['r33'] = F.relu(self.conv3_3(out['r32']))
        out['r34'] = F.relu(self.conv3_4(out['r33']))
        out['p3'] = self.pool3(out['r34'])
        out['r41'] = F.relu(self.conv4_1(out['p3']))
        out['r42'] = F.relu(self.conv4_2(out['r41']))
        out['r43'] = F.relu(self.conv4_3(out['r42']))
        out['r44'] = F.relu(self.conv4_4(out['r43']))
        out['p4'] = self.pool4(out['r44'])
        out['r51'] = F.relu(self.conv5_1(out['p4']))
        out['r52'] = F.relu(self.conv5_2(out['r51']))
        out['r53'] = F.relu(self.conv5_3(out['r52']))
        out['r54'] = F.relu(self.conv5_4(out['r53']))
        out['p5'] = self.pool5(out['r54'])
        return [out[key] for key in out_keys]

In [ ]:
def compute_cosine_similarity(content_features, style_features, centering = False, normalize_across_spatial_dims = False):
    """
    Compute cosine similarity between content and style features.
    
    Args:
        content_features: Content features (1, C, H_content, W_content)
        style_features: Style features (1, C, H_style, W_style)
        centering: Whether to center the features before computing similarity
        normalize_across_spatial_dims: Whether to normalize across spatial dimensions before computing similarity
    Returns:
        Cosine similarity matrix (H_content*W_content, H_style*W_style) 
    """
    content_features_flat = rearrange(content_features, "1 c h w -> c (h w)") # (C, H_content*W_content)
    style_features_flat = rearrange(style_features, "1 c h w -> c (h w)") # (C, H_style*W_style)

    if centering:
        content_mean = content_features_flat.mean(dim=1, keepdim=True)
        style_mean = style_features_flat.mean(dim=1, keepdim=True)
        content_features_flat = content_features_flat - content_mean
        style_features_flat = style_features_flat - style_mean

    if normalize_across_spatial_dims:
        content_norms = torch.norm(content_features_flat, dim=1, keepdim=True)  # (C, H_content*W_content)
        style_norms = torch.norm(style_features_flat, dim=1, keepdim=True)      # (C, H_style*W_style)

        # Normalize features by their norms to get unit vectors
        content_features_flat = content_features_flat / (content_norms + 1e-8)  # (C, H_content*W_content)
        style_features_flat = style_features_flat / (style_norms + 1e-8)        # (C, H_style*W_style)

    content_len = content_features_flat.shape[1]
    style_len = style_features_flat.shape[1]

    # Compute norms for cosine normalization
    content_norms = torch.norm(content_features_flat, dim=0, keepdim=True)  # (1, H_content*W_content)
    style_norms = torch.norm(style_features_flat, dim=0, keepdim=True)      # (1, H_style*W_style)

    # Normalize features by their norms to get unit vectors
    content_features_flat = content_features_flat / (content_norms + 1e-8)  # (C, H_content*W_content)
    style_features_flat = style_features_flat / (style_norms + 1e-8)        # (C, H_style*W_style)

    # If tensors are too large, compute similarity in smaller batches
    if content_len * style_len > 1e10:  # 100M elements threshold #NOTE: choose size based on available GPU memory
        # Use a more memory-efficient approach with manual batching
        similarities = []
        
        # Process in smaller batches to avoid memory issues
        batch_size = min(2048, content_len)  # Process 1000 at a time to reduce memory pressure
        
        for i in range(0, content_len, batch_size):
            print(f"Processing content features {i} to {min(i + batch_size, content_len)} / {content_len}")
            end_idx = min(i + batch_size, content_len)
            content_batch = content_features_flat[:, i:end_idx]  # (C, batch_size)
            
            # Compute similarity for this batch (dot product of normalized vectors)
            similarities_batch = torch.matmul(content_batch.T, style_features_flat)  # (batch_size, style_len)

            # move to cpu and store to reduce GPU memory pressure
            similarities.append(similarities_batch.cpu())
            
            # Clear the batch tensor to free GPU memory
            del similarities_batch, content_batch
            cleanup_gpu_memory()
        
        # Concatenate all similarities on CPU
        similarities = torch.cat(similarities, dim=0)
        
    else:
        # For smaller tensors, compute directly (dot product of normalized vectors)
        similarities = torch.matmul(content_features_flat.T, style_features_flat)  # (H_content*W_content, H_style*W_style)

    return similarities

def compute_cosine_similarity_all_nbhds(content_features, style_features, neighborhood_size = 3, centering = False, normalize_across_spatial_dims = False):
    """
    Compute cosine similarity between all neighborhoods of content and style features.
    
    Args:
        content_features: Content features (1, C, H_content, W_content)
        style_features: Style features (1, C, H_style, W_style)
        neighborhood_size: Size of neighborhoods to compare (default 3)
        centering: Whether to center the features before computing similarity
        normalize_across_spatial_dims: Whether to normalize across spatial dimensions (default False)
    Returns:
        Cosine similarity matrix (H_content*W_content, H_style*W_style) 
    """

    if centering:
        content_mean = repeat(reduce(content_features, "1 c h w -> 1 c", "mean"), "1 c -> 1 c h w", h = content_features.shape[2], w = content_features.shape[3])
        style_mean = repeat(reduce(style_features, "1 c h w -> 1 c", "mean"), "1 c -> 1 c h w", h = style_features.shape[2], w = style_features.shape[3])
        content_features = content_features - content_mean
        style_features = style_features - style_mean

    if normalize_across_spatial_dims:
        content_norms = torch.norm(content_features, dim=(2, 3), keepdim=True)  # (1, C, H_content, W_content)
        style_norms = torch.norm(style_features, dim=(2, 3), keepdim=True)      # (1, C, H_style, W_style)

        # Normalize features by their norms to get unit vectors
        content_features = content_features / (content_norms + 1e-8)  # (1, C, H_content, W_content)
        style_features = style_features / (style_norms + 1e-8)        # (1, C, H_style, W_style)

    content_len = content_features.shape[2] * content_features.shape[3]
    style_len = style_features.shape[2] * style_features.shape[3]

    # extract neighborhoods at each pixel as extra dimension
    content_features_nbhds = extract_neighborhoods(content_features, neighborhood_size) # (H_content, W_content, c * neighborhood_size^2)
    style_features_nbhds = extract_neighborhoods(style_features, neighborhood_size) # (H_style, W_style, c * neighborhood_size^2)

    # reshape for matrix multiplication
    content_features_nbhds = rearrange(content_features_nbhds, "h w c_nbhd -> (h w) c_nbhd") # (H_content*W_content, c * neighborhood_size^2)
    style_features_nbhds = rearrange(style_features_nbhds, "h w c_nbhd -> (h w) c_nbhd") # (H_style*W_style, c * neighborhood_size^2)

    # normalize across whole neighborhood for cosine normalization
    content_norms = torch.norm(content_features_nbhds, dim=1, keepdim=True)
    style_norms = torch.norm(style_features_nbhds, dim=1, keepdim=True)

    # Normalize features by their norms to get unit vectors
    content_features_nbhds = content_features_nbhds/ (content_norms + 1e-8)
    style_features_nbhds = style_features_nbhds / (style_norms + 1e-8)

    # If tensors are too large, compute similarity in smaller batches
    if content_len * style_len > 1e10:  # 100M elements threshold #NOTE: choose size based on available GPU memory
        # Use a more memory-efficient approach with manual batching
        similarities = []
        
        # Process in smaller batches to avoid memory issues
        batch_size = min(2048, content_len)  # Process 1000 at a time to reduce memory pressure
        
        for i in range(0, content_len, batch_size):
            print(f"Processing content features {i} to {min(i + batch_size, content_len)} / {content_len}")
            end_idx = min(i + batch_size, content_len)
            content_batch = content_features_nbhds[i:end_idx, :]  # (batch_size, C * nbhd_size^2)
            
            # Compute similarity for this batch (dot product of normalized vectors)
            similarities_batch = torch.matmul(content_batch, style_features_nbhds.T)  # (batch_size, style_len)

            # move to cpu and store to reduce GPU memory pressure
            similarities.append(similarities_batch.cpu())
            
            # Clear the batch tensor to free GPU memory
            del similarities_batch, content_batch
            cleanup_gpu_memory()
        
        # Concatenate all similarities on CPU
        similarities = torch.cat(similarities, dim=0)
        
    else:
        # For smaller tensors, compute directly (dot product of normalized vectors)
        similarities = torch.matmul(content_features_nbhds, style_features_nbhds.T)  # (H_content*W_content, H_style*W_style)

    return similarities

def extract_neighborhoods(features, neighborhood_size=3):
    """
    Extract neighborhood centered at a specific position.
    
    Args:
        features: Feature map (B, C, H, W)
        neighborhood_size: Size of square neighborhood (default 3)
        
    Returns:
        neighborhoods: Extracted neighborhoods (H, W, C * neighborhood_size * neighborhood_size)
    """
    B, C, H, W = features.shape
    pad = neighborhood_size // 2
    # pad with zeros to handle borders
    features = F.pad(features, (pad, pad, pad, pad), mode ='constant', value = 0)

    neighborhoods = torch.zeros(H, W, C * neighborhood_size * neighborhood_size, device=features.device)

    for row in range(H):
        for col in range(W):
            neighborhoods[row, col] = rearrange(features[:, :, row:row+neighborhood_size, col:col+neighborhood_size], "1 C h w -> (C h w)")
    
    return neighborhoods

def extract_neighborhood_at_position(features, position, neighborhood_size=3):
    """
    Extract neighborhood centered at a specific position.
    
    Args:
        features: Feature map (B, C, H, W)
        position: (row, col) tuple specifying center position
        neighborhood_size: Size of square neighborhood (default 3)
        
    Returns:
        neighborhood: Extracted neighborhood (B, C, neighborhood_size, neighborhood_size)
    """
    B, C, H, W = features.shape
    row, col = position
    pad = neighborhood_size // 2

    # pad with zeros to handle borders
    features = F.pad(features, (pad, pad, pad, pad), mode ='constant', value = 0)

    # extract neighborhood centered at (row, col)
    neighborhood = features[:, :, row:row+neighborhood_size, col:col+neighborhood_size]

    return neighborhood

def compute_single_neighborhood_similarity(content_neighborhood, style_neighborhood, L_shaped = False):
    """
    Compute cosine similarity between two neighborhoods.
    
    Args:
        content_neighborhood: Content neighborhood (B, C, H, W)
        style_neighborhood: Style neighborhood (B, C, H, W)
        L_shaped: Whether to use L-shaped neighborhood (default False)
        
    Returns:
        similarity: Cosine similarity value
    """

    # Flatten neighborhoods for similarity computation
    if L_shaped:
        # extract L-shaped neighborhood (top half and left half) and flatten
        neighborhood_size = content_neighborhood.shape[2]
        content_flat = torch.zeros(content_neighborhood.shape[0], content_neighborhood.shape[1], (neighborhood_size + 1) * (neighborhood_size // 2) + 1, device=content_neighborhood.device)
        style_flat = torch.zeros(style_neighborhood.shape[0], style_neighborhood.shape[1], (neighborhood_size + 1) * (neighborhood_size // 2) + 1, device=style_neighborhood.device)
        content_flat[:, :, :neighborhood_size * (neighborhood_size // 2)] = rearrange(content_neighborhood[:, :, :neighborhood_size // 2, :], "b c h w -> b c (h w)")
        content_flat[:, :, neighborhood_size * (neighborhood_size // 2):] = content_neighborhood[:, :, neighborhood_size // 2, :neighborhood_size // 2 + 1]
        style_flat[:, :, :neighborhood_size * (neighborhood_size // 2)] = rearrange(style_neighborhood[:, :, :neighborhood_size // 2, :], "b c h w -> b c (h w)")
        style_flat[:, :, neighborhood_size * (neighborhood_size // 2):] = style_neighborhood[:, :, neighborhood_size // 2, :neighborhood_size // 2 + 1]
        
        # nbhd_overlap = (content_flat[0,0] != 0).float() * (style_flat[0,0] != 0).float()

        content_flat = rearrange(content_flat, "b c hw -> (b c hw)")
        style_flat = rearrange(style_flat, "b c hw -> (b c hw)")

    else:
        # nbhd_overlap = (content_flat[0,0] != 0).float() * (style_flat[0,0] != 0).float()

        content_flat = rearrange(content_neighborhood, "b c h w -> (b c h w)")
        style_flat = rearrange(style_neighborhood, "b c h w -> (b c h w)")
        
    # Normalize over whole neighborhood for cosine similarity
    content_norm = torch.norm(content_flat) + 1e-8
    style_norm = torch.norm(style_flat) + 1e-8
    
    content_normalized = content_flat / content_norm
    style_normalized = style_flat / style_norm
    
    # Compute cosine similarity
    similarity = torch.sum(content_normalized * style_normalized) # scalar product of normalized vectors

    #NOTE: normalizing with overlap of nbhds does not seem to make a difference
    # similarity = similarity / (nbhd_overlap.sum() + 1e-8)  # Normalize by the number of overlapping elements

    return similarity.item()

def find_best_matches_initial(content_features, style_features, use_sampling = False, sampling_ratio = 0.1, centering = False, neighborhood_size=3, normalize_across_spatial_dims=False):
    """
    Find best matches at r51 with neighborhood-based similarity computation.
    
    Args:
        content_features: Feature map from content image (B, C, H_content, W_content)
        style_features: Feature map from style image (B, C, H_style, W_style)
        use_sampling: Whether to use sampling for matching (default False)
        sampling_ratio: Ratio of best matches to sample from (default 0.1)
        centering: Whether to center features before computing similarity
        neighborhood_size: Size of neighborhoods to compare (default 3)
        num_candidates: Number of candidate pixels to sample (default 100)
        normalize_across_spatial_dims: Whether to normalize across spatial dimensions to get unit vectors (default False)

    Returns:
        best_match_indices: Indices of best matching style features (H_content, W_content)
        best_match_vectors: Best matching style feature vectors (B, C, H_content, W_content)
    """
    # Safety check for empty tensors
    if content_features.numel() == 0 or style_features.numel() == 0:
        raise ValueError("Empty feature tensors provided to find_best_matches_initial")
    
    # Shape assertions for debugging
    assert content_features.dim() == 4, f"Expected content_features to be 4D, got {content_features.dim()}D"
    assert style_features.dim() == 4, f"Expected style_features to be 4D, got {style_features.dim()}D"
    assert content_features.size(0) == style_features.size(0), f"Batch sizes don't match: {content_features.size(0)} vs {style_features.size(0)}"
    assert content_features.size(1) == style_features.size(1), f"Channel dimensions don't match: {content_features.size(1)} vs {style_features.size(1)}"
    
    B, C, H_content, W_content = content_features.shape
    _, _, H_style, W_style = style_features.shape
    
    # Initialize lists to store results
    best_match_indices = []
    
    # compute all nbhds at once
    similarities = compute_cosine_similarity_all_nbhds(content_features, style_features, neighborhood_size, centering) # (H_content*W_content, H_style*W_style)

    
    # If using sampling for matching, sample from top matches based on similarity scores
    if use_sampling:
    
        num_top_matches = max(1, int(similarities.shape[1] * sampling_ratio))
        
        # For each content feature, sample from the top matches based on similarity scores
        best_match_indices = []
        
        # Process each content feature vector
        for i in range(similarities.shape[0]):
            # Get similarity scores for this content feature across all style features
            sim_scores = similarities[i]  # (H_style*W_style,)
            
            # Assertions for similarity scores
            assert sim_scores.shape[0] == H_style * W_style, f"Similarity scores don't match style features: {sim_scores.shape[0]} vs {H_style * W_style}"
            
            # Get indices of top matches (sorted by similarity score in descending order)
            top_indices = torch.topk(sim_scores, min(num_top_matches, len(sim_scores)), largest=True, sorted=True).indices
            
            # Sample one index from the top matches using probability proportional to similarity scores #NOTE: maybe we can just pick randomly from the top matches without weighting?
            # Convert similarity scores to probabilities (softmax)
            probs = F.softmax(sim_scores[top_indices], dim=0)
            
            # Sample one index from top matches based on probabilities
            sampled_idx = torch.multinomial(probs, 1).item()
            
            # Get the actual style feature index
            actual_idx = top_indices[sampled_idx].item()
            best_match_indices.append(actual_idx)

        # Convert to tensor
        best_match_indices = torch.tensor(best_match_indices, dtype=torch.long)

    else:
        # Find the index of the maximum similarity for each content feature vector
        best_match_indices = torch.argmax(similarities, dim=1).long() # (H_content*W_content,)

    # Clear the similarities tensor to free GPU memory
    del similarities
    cleanup_gpu_memory()

    # Move to GPU if available
    if torch.cuda.is_available():
        best_match_indices = best_match_indices.cuda()
    
    # Assertions for final indices
    assert best_match_indices.shape[0] == H_content * W_content, f"Indices don't match content features: {best_match_indices.shape[0]} vs {H_content * W_content}"

    # Find the best matching style feature vectors
    style_features_flat = rearrange(style_features, "1 c h w -> c (h w)") # (C, H_style*W_style)
    best_match_vectors = style_features_flat[:, best_match_indices]  # (C, H_content*W_content)
    best_match_vectors = rearrange(best_match_vectors, "c (h w) -> 1 c h w", h = H_content, w = W_content)  # (1, C, H_content, W_content)
    
    # Reshape the indices back to (H_content, W_content)
    best_match_indices = rearrange(best_match_indices, "(h w) -> h w", h=H_content, w=W_content)  # (H_content, W_content)
    
    # Assertions for final indices shape
    assert best_match_indices.shape == (H_content, W_content), \
        f"Final indices shape mismatch: {best_match_indices.shape} vs ({H_content}, {W_content})"

    return best_match_indices, best_match_vectors

def cleanup_gpu_memory():
    """Helper function to clean up GPU memory"""
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def get_candidate_position(synthesized_indices, row, col, row_shift, col_shift, H_style, W_style):
    """
    Get candidate style position based on shifted style position.
    
    Args:
        synthesized_indices: Indices of synthesized features (H_content, W_content)
        row: Row index of current content pixel
        col: Column index of current content pixel
        row_shift: Row shift for neighborhood
        col_shift: Column shift for neighborhood
        H_style: Height of style feature map
        W_style: Width of style feature map
    Returns:
        style_row: Row index of candidate style pixel
        style_col: Column index of candidate style pixel
    """
    synthesized_index = synthesized_indices[row, col].item()
    style_row = synthesized_index // W_style - row_shift
    style_col = synthesized_index % W_style - col_shift

    # only add valid style positions within bounds, otherwise sample position from style feature map
    if not (0 <= style_row < H_style and 0 <= style_col < W_style):
        style_row = np.random.randint(0, H_style)
        style_col = np.random.randint(0, W_style)

    return style_row, style_col

def get_candidate_positions(synthesized_indices, row, col, neighborhood_size, H_style, W_style):
    """
    Get candidate style positions based on synthesized indices and neighborhood.

    Args:
        synthesized_indices: Indices of synthesized features (H_content, W_content)
        row: Row index of current content pixel
        col: Column index of current content pixel
        neighborhood_size: Size of neighborhood to consider (default 3)
        H_style: Height of style feature map
        W_style: Width of style feature map
    Returns:
        candidate_positions: List of (style_row, style_col) tuples for candidate style pixels
    """

    # get candidate pixels from L neighborhood of upsampled index
    candidate_positions = []
    H_content, W_content = synthesized_indices.shape

    for dr in range(-neighborhood_size//2, 0): # compute indices from upper half of the neighborhood, do half of center line later
        for dc in range(-neighborhood_size//2, neighborhood_size//2 + 1):

            r = row + dr
            c = col + dc

            if 0 <= r < H_content and 0 <= c < W_content:

                # get candidate style position based on synthesized index and relative position to current pixel
                style_row, style_col = get_candidate_position(synthesized_indices, r, c, dr, dc, H_style, W_style)
                candidate_positions.append((style_row, style_col))

    for dc in range(-neighborhood_size//2, 1): # comput indices from the left half of the center line of the neighborhood

        r = row
        c = col + dc

        if 0 <= r < H_content and 0 <= c < W_content:

            # get candidate style position based on synthesized index and relative position to current pixel
            style_row, style_col = get_candidate_position(synthesized_indices, r, c, 0, dc, H_style, W_style)
            candidate_positions.append((style_row, style_col))

    return candidate_positions

def find_best_matches_weighted(content_features, style_features, synthesized_features, synthesized_indices, alpha=0.5, weight_field = None, 
                               centering=False, neighborhood_size=3, normalize_across_spatial_dims=False):
    """
    Find best matches at r41 and r31 using weighted neighborhood similarity.
    
    Args:
        content_features: Feature map from content image (B, C, H_content, W_content)
        style_features: Feature map from style image (B, C, H_style, W_style)
        synthesized_features: Previously synthesized features (B, C, H_content, W_content)
        synthesized_indices: Indices of synthesized features (H_content, W_content)
        alpha: Weight for content-style similarity vs synthesized-style similarity (alpha = 1.0 means only content-style similarity)
        weight_fields: Tensor that specifies spatially varying alpha values for given layer, if not None takes precedence over 'alpha'
        centering: Whether to center features before computing similarity
        neighborhood_size: Size of neighborhoods to compare (default 3)
        normalize_across_spatial_dims: Whether to normalize across spatial dimensions to get unit vectors (default False)

    Returns:
        best_match_indices: Indices of best matching style features (H_content, W_content)
        best_match_vectors: Best matching style feature vectors (B, C, H_content, W_content)
    """
    # Safety check for empty tensors
    if content_features.numel() == 0 or style_features.numel() == 0 or synthesized_features.numel() == 0:
        raise ValueError("Empty feature tensors provided to find_best_matches_weighted")
    
    # Shape assertions (for debugging)
    assert content_features.dim() == 4, f"Expected content_features to be 4D, got {content_features.dim()}D"
    assert style_features.dim() == 4, f"Expected style_features to be 4D, got {style_features.dim()}D"
    assert synthesized_features.dim() == 4, f"Expected synthesized_features to be 4D, got {synthesized_features.dim()}D"
    assert content_features.size(0) == style_features.size(0), f"Batch sizes (content vs style) don't match: {content_features.size(0)} vs {style_features.size(0)}"
    assert synthesized_indices.dim() == 2, f"Expected synthesized_indices to be 2D, got {synthesized_indices.dim()}D"
    assert content_features.size(1) == style_features.size(1), f"Channel dimensions (content vs style) don't match: {content_features.size(1)} vs {style_features.size(1)}"
    assert synthesized_features.size(0) == style_features.size(0), f"Batch sizes (synthesized vs style) don't match: {synthesized_features.size(0)} vs {style_features.size(0)}"
    assert synthesized_features.size(1) == style_features.size(1), f"Channel dimensions (synthesized vs style) don't match: {synthesized_features.size(1)} vs {style_features.size(1)}"
    assert synthesized_features.size(2) == content_features.size(2) and synthesized_features.size(3) == content_features.size(3), \
        f"Synthesized features spatial dimensions don't match content features: {synthesized_features.size(2)}x{synthesized_features.size(3)} vs {content_features.size(2)}x{content_features.size(3)}"
    assert synthesized_indices.shape == (content_features.size(2), content_features.size(3)), \
        f"Synthesized indices shape mismatch: {synthesized_indices.shape} vs ({content_features.size(2)}, {content_features.size(3)})"

    B, C, H_content, W_content = content_features.shape
    _, _, H_style, W_style = style_features.shape
    
    # compute all nbhd similarities at once, we can compute content-style similarities at candidate positions beforehand since they don't change across iterations
    content_style_similarities = compute_cosine_similarity_all_nbhds(content_features, style_features, neighborhood_size, centering, normalize_across_spatial_dims) # (H_content*W_content, H_style*W_style)

    # center style features for cosine comparison
    if centering:
        style_mean = repeat(reduce(style_features, "1 c h w -> 1 c", "mean"), "1 c -> 1 c h w", h = style_features.shape[2], w = style_features.shape[3])
        style_features_centered = style_features - style_mean
    else:
        style_features_centered = style_features

    if normalize_across_spatial_dims:
        style_norms = torch.norm(style_features_centered, dim=(2, 3), keepdim=True)      # (1, C, H_style, W_style)
        style_features_centered = style_features_centered / (style_norms + 1e-8)

    # center synthesized features for cosine comparison
    if centering:
        synthesized_mean = repeat(reduce(synthesized_features, "1 c h w -> 1 c", "mean"), "1 c -> 1 c h w", h = synthesized_features.shape[2], w = synthesized_features.shape[3])
        synthesized_features_centered = synthesized_features - synthesized_mean
    else:
        synthesized_features_centered = synthesized_features

    # go through pixels in scanline order and sample candidates based on upsampled indices
    for row in tqdm(range(H_content)):
        for col in range(W_content):

            if normalize_across_spatial_dims:
                synthesized_norms = torch.norm(synthesized_features_centered, dim=(2, 3), keepdim=True)      # (1, C, H_content, W_content)
                synthesized_features_centered = synthesized_features_centered / (synthesized_norms + 1e-8)

            # compute candidate style positions based on synthesized indices and neighborhood
            candidate_positions = get_candidate_positions(synthesized_indices, row, col, neighborhood_size, H_style, W_style)

            # remove duplicate candidate positions
            candidate_positions = list(set(candidate_positions))

            # compute synthesized-style similarities at candidate positions
            synthesized_style_similarities_at_candidates = torch.zeros(len(candidate_positions), device=content_features.device)
            synthesized_features_nbhd = extract_neighborhood_at_position(synthesized_features_centered, (row, col), neighborhood_size) # (1, C, neighborhood_size, neighborhood_size)
            for i, candidate_pos in enumerate(candidate_positions):
                synthesized_style_similarities_at_candidates[i] = compute_single_neighborhood_similarity(synthesized_features_nbhd,
                                                                                         extract_neighborhood_at_position(style_features_centered, candidate_pos, neighborhood_size), L_shaped = True) # (1, 1)

            # compare similarities (content vs style and synthesized vs style) at candidate positions
            content_style_similarities_at_candidates = content_style_similarities[row * W_content + col, [pos[0] * W_style + pos[1] for pos in candidate_positions]] # (1, len(candidate_positions))

            # Weighted combination
            if weight_field is None:
                similarity = alpha * content_style_similarities_at_candidates + (1.0 - alpha) * synthesized_style_similarities_at_candidates # (1, len(candidate_positions))
            else:
                similarity = weight_field[0, 0, row, col] * content_style_similarities_at_candidates + (1.0 - weight_field[0, 0, row, col]) * synthesized_style_similarities_at_candidates # (1, len(candidate_positions))

            # determine best candidate based on weighted similarity
            best_candidate_idx = torch.argmax(similarity).item()
            best_candidate_pos = candidate_positions[best_candidate_idx]

            # update synthesized index for this pixel to best candidate position
            synthesized_indices[row, col] = best_candidate_pos[0] * W_style + best_candidate_pos[1]

            # update synthesized feature map
            synthesized_features[:, :, row, col] = style_features[:, :, best_candidate_pos[0], best_candidate_pos[1]]

    return synthesized_indices, synthesized_features

def upsample_index_array(index_array, target_shape, style_spatial_shape, previous_style_spatial_shape):
    """
    Upsample index array to target shape using nearest neighbor interpolation.
    
    Args:
        index_array: Index array from previous layer (H, W)
        target_shape: Target shape (H_target, W_target)
        style_spatial_shape: Style spatial shape (H_style, W_style)
        previous_style_spatial_shape: Previous style spatial shape (H_prev, W_prev)
        
    Returns:
        upsampled_indices: Upsampled index array (H_target, W_target)
    """
    # Safety check for empty arrays
    if index_array.numel() == 0:
        raise ValueError("Empty index array provided to upsample_index_array")
    
    # Convert index array to tensor if needed
    if not isinstance(index_array, torch.Tensor):
        index_array = torch.tensor(index_array)
    
    # Ensure the index array is on the correct device
    if torch.cuda.is_available():
        index_array = index_array.cuda()
    
    # Reshape to 4D tensor for interpolation: (1, 1, H, W)
    index_tensor = rearrange(index_array, "h w -> 1 1 h w").float()  # (1, 1, H, W)
    
    # Safety check for valid target shape
    if target_shape[0] <= 0 or target_shape[1] <= 0:
        raise ValueError(f"Invalid target shape: {target_shape}")
    
    # Upsample using nearest neighbor interpolation
    upsampled_tensor = F.interpolate(index_tensor, size=target_shape, mode='nearest')

    # adjust entries (indices) to new style features shape
    rows, cols = torch.unravel_index(upsampled_tensor.long(), previous_style_spatial_shape)  # (1, 1, H_target, W_target) each

    # upscale indices to match new style spatial shape
    scale_h = style_spatial_shape[0] / previous_style_spatial_shape[0]
    scale_w = style_spatial_shape[1] / previous_style_spatial_shape[1]
    rows = (rows.float() * scale_h).long()
    cols = (cols.float() * scale_w).long()

    # clamp indices to be within bounds of style spatial shape
    rows = torch.clamp(rows, 0, style_spatial_shape[0] - 1)  # row index
    cols = torch.clamp(cols, 0, style_spatial_shape[1] - 1)  # column index

    # ravel indices back to flat indices for gathering
    upsampled_tensor = rows * style_spatial_shape[1] + cols  # (1, 1, H_target, W_target)
    
    # Remove extra dimensions and return as integer array
    upsampled_indices = rearrange(upsampled_tensor, "1 1 h w -> h w").long()  # (H_target, W_target)
    
    return upsampled_indices

def gather_style_features_at_indices(style_features, indices):
    """
    Gather style features at specific indices.
    
    Args:
        style_features: Style features (B, C, H_style, W_style)
        indices: Index array (H_content, W_content) indicating which features to gather
        
    Returns:
        gathered_features: Features gathered at specified indices (B, C, H_content, W_content)
    """
    # Safety check for empty tensors
    if style_features.numel() == 0 or indices.numel() == 0:
        raise ValueError("Empty tensors provided to gather_style_features_at_indices")
    
    # Get spatial dimensions
    B, C, H, W = style_features.shape
    
    # Flatten style features for easier indexing
    style_features_flat = rearrange(style_features, "b c h w -> b c (h w)") # (B, C, H_style*W_style)
    
    # Convert indices to flattened indices
    # Assuming indices are in (H_content, W_content) format, convert to linear indices
    indices_flat = indices.view(-1)  # (H_content*W_content,)
    
    # Make sure indices are within bounds
    indices_flat = torch.clamp(indices_flat, 0, style_features_flat.shape[2] - 1)
    
    # Safety check to prevent excessive memory allocation
    if indices_flat.numel() > 10000000:  # Limit to 10M elements
        raise ValueError("Too many indices requested - potential memory allocation issue")
    
    # Gather features at specified indices
    gathered_features_flat = style_features_flat[:, :, indices_flat] # (B, C, H_content*W_content)
    
    # Reshape back to original format
    gathered_features = gathered_features_flat.view(B, C, indices.shape[0], indices.shape[1])
    
    return gathered_features

def synthesize_features(content_features_list, style_features_list, 
                                        alpha=0.5, weight_fields = None, use_sampling=False, sampling_ratio=0.1, centering = False, neighborhood_size=3, normalize_across_spatial_dims=False):
    """
    Complete multi-stage feature synthesis with advanced approach:
    r51 (greedy) -> r41 (weighted) -> r31 (weighted) with index upsampling.
    
    Args:
        content_features_list: List of content features at different layers
        style_features_list: List of style features at different layers
        layer_names: List of layer names in order (deepest to shallowest)
        alpha: Weight for weighted loss, alpha = 1.0 means only content-style similarity
        weight_fields: List of tensors that specify spatially varying alpha values for each layer, if not None takes precedence over 'alpha'
        use_greedy: Whether to use greedy search for the deepest layer
        sampling_ratio: Ratio of best matches to sample from for r51
        centering: Whether to center features before computing similarities
        neighborhood_size: Size of neighborhood blocks to consider for similarity (default 3)
        normalize_across_spatial_dims: Whether to normalize across spatial dimensions to get unit vectors (default False)
        
    Returns:
        target_features_list: Final target features at same levels as content/style features
    """
    # Process from deepest layer to shallowest
    synthesized_features_list = []
    match_indices = None
    previous_style_spatial_shape = None

    for layer_idx in range(len(content_features_list)):
        print(f"Synthesizing layer {layer_idx}")
        content_features = content_features_list[layer_idx]
        style_features = style_features_list[layer_idx]
        
        # For the deepest layer (r51), use greedy search. For intermediate layers, use weighted loss with upsampled indices.
        if layer_idx == 0:

            match_indices, synthesized_features = find_best_matches_initial(
                content_features, style_features, use_sampling, sampling_ratio, centering,
                neighborhood_size=neighborhood_size, normalize_across_spatial_dims=normalize_across_spatial_dims
            )

        else:
        # "upsample" indices from previous layer and gather features if not the first layer

            # Upsample the previous indices to current layer resolution
            content_spatial_shape = content_features.shape[2:4]  # (H, W) of current layer
            style_spatial_shape = style_features.shape[2:4]  # (H, W) of current layer
            upsampled_indices = upsample_index_array(match_indices, content_spatial_shape, style_spatial_shape, previous_style_spatial_shape)
            
            # Gather style features at upsampled indices
            synthesized_features = gather_style_features_at_indices(style_features, upsampled_indices)

            if weight_fields is not None:
                weight_field = weight_fields[layer_idx]
            else:
                weight_field = None

            # Use the gathered features to find best matches with weighted loss
            match_indices, synthesized_features = find_best_matches_weighted(
                content_features, style_features, synthesized_features, upsampled_indices, alpha, weight_field, centering,
                neighborhood_size=neighborhood_size, normalize_across_spatial_dims=normalize_across_spatial_dims
            )

        previous_style_spatial_shape = style_features.shape[2:4]  # (H, W) of style features at current layer

        synthesized_features_list.append(synthesized_features)
        
    # The final synthesized features at become our target features
    # Ensure the tensor is properly shaped and on the correct device
    if torch.cuda.is_available():
       synthesized_features_list = [feat.cuda() for feat in synthesized_features_list] 
    return synthesized_features_list

In [ ]:
#get network
vgg = VGG()
vgg_dict_sanet = torch.load(os.path.join(model_dir, 'vgg_normalised.pth'), weights_only=True)
# translate state dict from sanet
translation_dict = {
    "2.weight" : "conv1_1.weight",
    "2.bias" : "conv1_1.bias",
    "5.weight" : "conv1_2.weight",
    "5.bias" : "conv1_2.bias",
    "9.weight" : "conv2_1.weight",
    "9.bias" : "conv2_1.bias",
    "12.weight" : "conv2_2.weight",
    "12.bias" : "conv2_2.bias",
    "16.weight" : "conv3_1.weight",
    "16.bias" : "conv3_1.bias",
    "19.weight" : "conv3_2.weight",
    "19.bias" : "conv3_2.bias",
    "22.weight" : "conv3_3.weight",
    "22.bias" : "conv3_3.bias",
    "25.weight" : "conv3_4.weight",
    "25.bias" : "conv3_4.bias",
    "29.weight" : "conv4_1.weight",
    "29.bias" : "conv4_1.bias",
    "32.weight" : "conv4_2.weight",
    "32.bias" : "conv4_2.bias",
    "35.weight" : "conv4_3.weight",
    "35.bias" : "conv4_3.bias",
    "38.weight" : "conv4_4.weight",
    "38.bias" : "conv4_4.bias",
    "42.weight" : "conv5_1.weight",
    "42.bias" : "conv5_1.bias",
    "45.weight" : "conv5_2.weight",
    "45.bias" : "conv5_2.bias",
    "48.weight" : "conv5_3.weight",
    "48.bias" : "conv5_3.bias",
    "51.weight" : "conv5_4.weight",
    "51.bias" : "conv5_4.bias"
}
vgg_dict = dict((translation_dict.get(key, key), value) for (key, value) in vgg_dict_sanet.items())
#NOTE: the vgg from SANET has a 1x1 conv. layer to do what is done here with preprocessing (prep function)
# -> remove keys, 0.weight and 0.bias
del vgg_dict["0.weight"]
del vgg_dict["0.bias"]

vgg.load_state_dict(vgg_dict)
for param in vgg.parameters():
    param.requires_grad = False
if torch.cuda.is_available():
    vgg.cuda()

## Choose content image, style image and scalar field

In [ ]:
# load images, ordered as [style_image, content_image]
style_img_names = ['mondrian.jpg']
content_img_names = ['471.jpg']
scalar_field_name = "gradient_for_471.jpg"

max_style_size = 512
max_content_size = 512

mask = "none" # can use "none", "scalar_field", "binary" and "gradient"

downsampling_type = "minpool" # "minpool", "bilinear" and "receptive"
downsampling_use_relu = False # whether to use relu layers in the downsampling network

remap_colors = True

use_content_mask = False
content_mask_path = os.path.join(base_dir, "<content_mask_path>")

style_imgs = [Image.open(os.path.join(style_dir, name)) for name in style_img_names]
content_imgs = [Image.open(os.path.join(content_dir, name)) for name in content_img_names]
scalar_field_img = Image.open(os.path.join(scalar_field_dir, scalar_field_name)) 

if use_content_mask:
    content_mask_img = Image.open(content_mask_path) 
    content_mask_img = resize_to_max_size(content_mask_img.convert("L"), max_content_size, multiple_of_16 = False)
    content_mask = np.array(content_mask_img).astype(np.bool) # mask is binary
else:
    content_mask = None

# color remapping
if remap_colors:
    color_remapper = LuminanceRemapper(resize_to_max_size(content_imgs[0], max_content_size, multiple_of_16 = False), content_mask = content_mask)
    style_imgs = [color_remapper.shift(style_img) for style_img in style_imgs]

In [ ]:
# resize content image to given size
preprocess_style = preprocess_image(max_style_size, multiple_of_16 = False) #NOTE: need multiple of 16 when masking style
preprocess_content = preprocess_image(max_content_size, multiple_of_16 = False)

style_imgs_torch = [preprocess_style(img) for img in style_imgs]
content_imgs_torch = [preprocess_content(img) for img in content_imgs]

init_img_torch = content_imgs_torch[0]

if torch.cuda.is_available():
    style_imgs_torch = [Variable(img.unsqueeze(0).cuda()) for img in style_imgs_torch]
    content_imgs_torch = [Variable(img.unsqueeze(0).cuda()) for img in content_imgs_torch]
    init_img_torch = init_img_torch.unsqueeze(0).cuda()
else:
    style_imgs_torch = [Variable(img.unsqueeze(0)) for img in style_imgs_torch]
    content_imgs_torch = [Variable(img.unsqueeze(0)) for img in content_imgs_torch]
    init_img_torch = init_img_torch.unsqueeze(0)
    print("using cpu as cuda is not available")
content_image = content_imgs_torch[0]

# for img in style_imgs_torch:
#     print("style image size: ", img.shape)
# print("content image size: ", content_image.shape)

# opt_img = Variable(torch.randn(content_image.size()).type_as(content_image.data), requires_grad=True) #NOTE: random init
opt_img = Variable(init_img_torch.data.clone(), requires_grad=True)

## Process / generate scalar field

In [ ]:
scalar_fields = []

# no mask
if mask == "none":
    scalar_field = torch.ones(1, 1, content_image.shape[-2], content_image.shape[-1])

# binary mask
if mask == "binary":
    scalar_field = 1e-9 * torch.ones(1, 1, content_image.shape[-2], content_image.shape[-1])
    scalar_field[:, :, :, content_image.shape[3]//2:] = 1.0

# soft mask
if mask == "gradient":
    scalar_field = torch.linspace(1.0, 1e-9, content_image.shape[-1])
    # scalar_field = torch.linspace(1e-9, 1.0, content_image.shape[-1])
    scalar_field = repeat(scalar_field, "w -> 1 1 h w", h = content_image.shape[-2])

# greyscale mask
if mask == "scalar_field":
    preprocess_greyscale_mask = preprocess_greyscale_image(max_content_size, multiple_of_16=False)
    scalar_field = preprocess_greyscale_mask(scalar_field_img.convert("L"))
    # fix normalization of scalar field
    if use_content_mask:
        content_mask_torch = rearrange(torch.from_numpy(content_mask), "h w -> 1 h w")
        scalar_field_max = scalar_field[content_mask_torch].max()
        scalar_field_min = scalar_field[content_mask_torch].min()
        scalar_field = (scalar_field - scalar_field_min) / (scalar_field_max - scalar_field_min)
    else:
        scalar_field = (scalar_field - scalar_field.min()) / (scalar_field.max() - scalar_field.min())
    scalar_field = torch.clip(scalar_field, 1e-9, 1.0) 
    scalar_field = rearrange(scalar_field, "c h w -> 1 c h w")

scalar_fields.append(scalar_field)

scalar_field_imgs = [transforms.functional.to_pil_image(repeat(rearrange(scalar_field, "1 1 h w -> 1 h w"), "1 h w -> c h w", c=3), mode = "RGB") for scalar_field in scalar_fields]
# move scalar field to gpu
scalar_fields = [scalar_field.cuda() for scalar_field in scalar_fields]

In [ ]:
#display images
for img in style_imgs:
    plt.imshow(img);plt.show()
for img in content_imgs:
    plt.imshow(img);plt.show()
for img in scalar_field_imgs:
    plt.imshow(img);plt.show()

## Set up nst and optimization parameters

In [ ]:
# NST parameters
alpha = 0.0 # Weight for content-style similarity (alpha = 1.0, E_c in paper) vs synthesized-style similarity (alpha = 0.0, E_s in paper)
use_sampling = True  # Use sampling for deepest layer instead of best matches
centering = True # Whether to center features before computing similarity
neighborhood_size = 5 # Size of neighborhood blocks to compare (default 3)
normalize_across_spatial_dims = False # Whether to normalize features across spatial dimensions

use_our_strength_control = False

if use_our_strength_control:
    #NOTE: our method 

    mask_loss = True #NOTE: mask loss using scalar field
    interpolate_target = True #NOTE: interpolate between stylized target features and content features using scalar field
    use_weight_field = False 

else:
    #NOTE: original paper

    mask_loss = False
    interpolate_target = False
    # use inverted scalar field as weight field for alpha value, as done in original paper
    use_weight_field = True 

if mask_loss or interpolate_target:
    alpha = 1.0 #NOTE: use this value for regions with full style strength

# Choose optimizer: 'adam' or 'lbfgs'
optimizer_type = 'adam'  # Change to 'adam' to use Adam optimizer
learning_rate = 0.05
max_iter = 2000
show_iter = 10

#define layers, loss functions, weights and compute optimization targets
synthesis_layers = ['r51', 'r41', 'r31']
# loss_layers = ['r51', 'r41', 'r31']
loss_layers = ['r31'] # not 100% clear from paper ...
all_layers = list(set(loss_layers + synthesis_layers))
# total_variation_layers = ['r11']

## dowsample scalar field

In [ ]:
# downsample content masks
resized_scalar_fields = []
for scalar_field in scalar_fields:
    if use_weight_field:
        resized_scalar_fields = [A.detach() for A in resize_scalar_field(scalar_field, all_layers)]
    else:
        if downsampling_type == "minpool":
            resized_scalar_fields = [A.detach() for A in minpool_resize_scalar_field(scalar_field, all_layers, use_relu = downsampling_use_relu)]
        elif downsampling_type == "receptive":
            resized_scalar_fields = [A.detach() for A in receptive_resize_scalar_field(scalar_field, all_layers, use_relu = downsampling_use_relu)]
        elif downsampling_type == "bilinear":
            resized_scalar_fields = [A.detach() for A in resize_scalar_field(scalar_field, all_layers)]
        else:
            raise ValueError(f"Unknown downsampling type = {downsampling_type}, use one of ['minpool', 'receptive', 'bilinear']")

resized_scalar_fields_dict = dict(zip(all_layers, resized_scalar_fields)) # create dict to easily access scalar fields by layer name

# plot resized scalar fields
for layer_idx, tensor in enumerate(resized_scalar_fields):
    img = tensor[0,0].detach().cpu()
    img = transforms.functional.to_pil_image(repeat(img, "h w -> c h w", c=3), mode = "RGB")
    print(all_layers[layer_idx])
    plt.imshow(img);plt.show()

## Run style transfer

In [ ]:
#run style transfer

opt_img = Variable(init_img_torch.data.clone(), requires_grad=True) #NOTE: if we want to start with new image
# opt_img = Variable(torch.randn(content_image.size()).type_as(content_image.data), requires_grad=True) #NOTE: random init
style_names = [img_name.split(".")[0] for img_name in style_img_names]
content_names = [img_name.split(".")[0] for img_name in content_img_names]
output_name = "images/output/deepfeaturesynthesis/"

if mask != "none" and mask_loss and interpolate_target:
    output_name += "ours/"
elif use_weight_field:
    output_name += "original/"

output_name += f"style={style_names}_content={content_names}_size={max_content_size}"
if remap_colors:
    output_name += "_remap_colors"
if mask_loss or interpolate_target or use_weight_field:
    if mask == "scalar_field":
        output_name += f"_scalar_field={scalar_field_name}"
    elif mask != "none": 
        output_name += "_" + mask 
if mask != "none":
    if downsampling_type == "bilinear":
        output_name += "_bilinear_downsampling"
    elif downsampling_type == "receptive":
        output_name += "_receptive_downsampling"
    if downsampling_use_relu:
        output_name += "_with_relu"
output_name += f"_alpha={alpha:.1f}"
output_name += "_nbhd_size=" + str(neighborhood_size)
output_name += "_sampling" if use_sampling else ""
output_name += "_centering" if centering else ""
output_name += "_normalize_spatial_dims" if normalize_across_spatial_dims else ""
output_name += f"_loss={'+'.join(loss_layers)}"
output_name += f"_{optimizer_type}_lr={learning_rate}_steps={max_iter}"

if interpolate_target:
    output_name += f"_interpolate_target"
if mask_loss:
    output_name += f"_mask_loss"

# compute content and style features

content_features_list = [A.detach() for A in vgg(content_image, synthesis_layers)]

content_features_dict = dict(zip(synthesis_layers, content_features_list))
style_features_list = []
for style_img in style_imgs_torch:
    style_features_list.extend(vgg(style_img, synthesis_layers))
style_features_list = [A.detach() for A in style_features_list]

# Generate target features using multi-stage neural-neighbor process

# use inverted scalar fields as weight fields
if use_weight_field:
    weight_fields = [1.0 - resized_scalar_fields_dict[layer] for layer in synthesis_layers]
else:
    weight_fields = None

target_features_list = synthesize_features(
    content_features_list, style_features_list, 
    alpha,
    weight_fields = weight_fields, # weight field for spatially varying alpha value
    use_sampling = use_sampling,
    sampling_ratio = 0.1,  # Sampling ratio for greedy search at r51
    centering = centering,  # Whether to center the features before computing similarity
    neighborhood_size = neighborhood_size,  # Size of neighborhood blocks to compare
    normalize_across_spatial_dims = normalize_across_spatial_dims  # Whether to normalize across spatial dimensions
)

# Ensure target features are on the correct device and detach from computation graph
if torch.cuda.is_available():
    target_features_list = [target_features.cuda() for target_features in target_features_list]
target_features_list = [target_features.detach() for target_features in target_features_list]

# create dict to easily access target features by layer name
target_features_dict = dict(zip(synthesis_layers, target_features_list))

# interpolate target and content features with scalar fields
if interpolate_target:
    for layer in loss_layers:
        target_features_dict[layer] = target_features_dict[layer] * resized_scalar_fields_dict[layer] + content_features_dict[layer] * (1.0 - resized_scalar_fields_dict[layer])

# Set up optimizer
if optimizer_type == 'adam':
    optimizer = optim.Adam([opt_img], lr=learning_rate)
else:  # default to LBFGS
    optimizer = optim.LBFGS([opt_img])

# set up loss function
if mask_loss:
    loss_function = MaskedMSELoss()
else:
    loss_function = nn.MSELoss()

n_iter=[0]

def closure():
    optimizer.zero_grad()
    out_features_list = vgg(opt_img, loss_layers) # list of features from loss_layers

    out_features_dict = dict(zip(loss_layers, out_features_list)) # create dict to easily access output features by layer name
    # out_features_for_total_variation_loss = out[len(style_layers) + len(content_layers):]
    
    if mask_loss:
        losses = [loss_function(out_features_dict[layer], target_features_dict[layer], resized_scalar_fields_dict[layer]) for layer in loss_layers]
    else:
        losses = [loss_function(out_features_dict[layer], target_features_dict[layer]) for layer in loss_layers]
    
    # total_variation_loss = torch.stack(total_variation_losses).sum()
    
    # Final combined loss
    loss = torch.stack(losses).sum() #+ total_variation_loss * total_variation_weights[0]
    loss.backward()

    #NOTE: gradient masking, can help with areas where no change should be made
    # opt_img.grad *= scalar_fields[0]

    n_iter[0]+=1
    if n_iter[0]%show_iter == (show_iter-1):
        print('Iteration: %d, loss: %f'%(n_iter[0]+1, loss.item()))
    return loss

while n_iter[0] <= max_iter:
    optimizer.step(closure)
    
#display result
out_img = postp(opt_img.data[0].cpu().squeeze())

# color remapping
if remap_colors:
    out_img = color_remapper.unshift(out_img)

plt.imshow(out_img)
plt.gcf().set_size_inches(10,10)

# save image
out_img.save(os.path.join(base_dir, output_name + ".png"), "PNG")